In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

import mlflow
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

/home/guslc/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def _load_env_for_this_notebook():
    """Load llm/api/.env even when Jupyter cwd is not llm/api (common cause of NoCredentialsError)."""
    candidates = []
    vf = globals().get("__vsc_ipynb_file__")
    if vf:
        candidates.append(Path(vf).resolve().parent / ".env")
    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd / ".env",
            cwd / "llm" / "api" / ".env",
            cwd / "api" / ".env",
        ]
    )
    for parent in [cwd, *cwd.parents]:
        nested = parent / "llm" / "api" / ".env"
        if nested.is_file():
            candidates.append(nested)
            break
    seen = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if p.is_file():
            # Shell/Jupyter may already set MLFLOW_S3_ENDPOINT_URL (old :9000); .env must win.
            load_dotenv(p, override=True)
            return


_load_env_for_this_notebook()

# Localhost :9000 is often not MinIO (ClickHouse etc.). docker-compose publishes MinIO at MINIO_API_HOST_PORT (default 19000).
_api_port = os.environ.get("MINIO_API_HOST_PORT", "19000")
_ep = os.environ.get("MLFLOW_S3_ENDPOINT_URL", "")
if _ep.startswith(("http://127.0.0.1:9000/", "http://127.0.0.1:9000", "http://localhost:9000/", "http://localhost:9000")):
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://127.0.0.1:{_api_port}"

params = {
    "model_name": "distilbert-base-uncased",
    "learning_rate": 5e-5,
    "batch_size": 16,
    "num_epochs": 3,
    "dataset_name": "ag_news",
    "task_name": "text_classification10",
    "seed": 42,
    "log_steps": 100,
    "max_seq_length": 128,
    "output_dir": "models/distilbert-base-uncased-ag-news",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

# Basic auth for mlflow[auth]: set MLFLOW_TRACKING_USERNAME / MLFLOW_TRACKING_PASSWORD in llm/api/.env (same as MLFLOW_TRACKING_* on the mlflow container).
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment(params["task_name"])




2026/05/07 19:56:51 INFO mlflow.tracking.fluent: Experiment with name 'text_classification10' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow/7', creation_time=1778194611699, experiment_id='7', last_update_time=1778194611699, lifecycle_stage='active', name='text_classification10', tags={}, trace_location=None, workspace='default'>

In [5]:
dataset = load_dataset(params["dataset_name"])
tokenizer = DistilBertTokenizer.from_pretrained(params["model_name"])

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=params["max_seq_length"])


train_dataset = dataset['train'].shuffle().select(range(20_000)).map(tokenize, batched=True)
test_dataset = dataset['test'].shuffle().select(range(2_000)).map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

labels = dataset['train'].features['label'].names



Map: 100%|██████████| 2000/2000 [00:00<00:00, 3723.19 examples/s]


In [6]:
train_dataset.to_parquet('data/train_dataset.parquet')
test_dataset.to_parquet('data/test_dataset.parquet')

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 19.75ba/s]


1787973

In [7]:
load_dotenv(Path.cwd() / ".env")

True

In [8]:
# # S3 artifact store (see experiment artifact_location): boto3 needs credentials.
# # Set AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION (and AWS_SESSION_TOKEN if temp creds),
# # or use `aws sso login` / ~/.aws/credentials. Optional: put them in .env (loaded in the params cell).
mlflow.log_artifact("data/train_dataset.parquet", artifact_path="datasets")
mlflow.log_artifact("data/test_dataset.parquet", artifact_path="datasets")

In [9]:
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)

In [10]:
labels = dataset['train'].features['label'].names

In [11]:
labels

['World', 'Sports', 'Business', 'Sci/Tech']

In [12]:
model = DistilBertForSequenceClassification.from_pretrained(params["model_name"], num_labels=len(labels))
model.config.id2label = {i: label for i, label in enumerate(labels)}

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1777.38it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
print(torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = AdamW(model.parameters(), lr=params["learning_rate"])

def evaluate(model, dataloader, device):
    model.eval()
    preds = []
    labels = []
    with torch.no_grad():
        for batch in dataloader:
            inputs, masks, batch_labels = (
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["label"].to(device),
            )

            outputs = model(inputs, attention_mask=masks)
            logits = outputs.logits
            _, pred_ids = torch.max(logits, dim=1)

            preds.extend(pred_ids.cpu().numpy().tolist())
            labels.extend(batch_labels.cpu().numpy().tolist())
            
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    
    return accuracy, precision, recall, f1


False


In [14]:
# from mlflow.tracking import MlflowClient

# # Attach training to this existing run (display name in MLflow UI).
# MLFLOW_RUN_NAME = "salty-smelt-664"

# _mlfc = MlflowClient()
# _exp_id = _mlfc.get_experiment_by_name(params["task_name"]).experiment_id
# _target_run_id = None
# for _r in _mlfc.search_runs(experiment_ids=[_exp_id], max_results=500):
#     _nm = getattr(_r.info, "run_name", None) or _r.data.tags.get("mlflow.runName", "")
#     if _nm == MLFLOW_RUN_NAME:
#         _target_run_id = _r.info.run_id
#         break
# if _target_run_id is None:
#     raise ValueError(
#         f"No run named {MLFLOW_RUN_NAME!r} in experiment {params['task_name']!r}. "
#         "Create it in the UI or start a run with that name first."
#     )

# if mlflow.active_run() is not None:
#     mlflow.end_run()

# with mlflow.start_run(run_id=_target_run_id) as run:
#     mlflow.log_params(params)

with tqdm(total=params['num_epochs'] * len(train_loader), desc=f"Epoch [1/{params['num_epochs']}] - (Loss: N/A) - Steps") as pbar:
    for epoch in range(params['num_epochs']):
        running_loss = 0.0
        for i, batch in enumerate(train_loader, 0):
            inputs, masks, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs, attention_mask=masks, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i and i % params['log_steps'] == 0:
                avg_loss = running_loss / params['log_steps']

                pbar.set_description(f"Epoch [{epoch + 1}/{params['num_epochs']}] - (Loss: {avg_loss:.3f}) - Steps")
                mlflow.log_metric("loss", avg_loss, step=epoch * len(train_loader) + i)
                
                running_loss = 0.0
            pbar.update(1)

        # Evaluate Model
        accuracy, precision, recall, f1 = evaluate(model, test_loader, device)
        print(f"Epoch {epoch + 1} Metrics: Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
        
        mlflow.log_metrics({'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}, step=epoch)

os.makedirs(params['output_dir'], exist_ok=True)
model.save_pretrained(params['output_dir'])
tokenizer.save_pretrained(params['output_dir'])

mlflow.log_artifacts(params['output_dir'], artifact_path="model")


print('Finished Training')

Epoch [1/3] - (Loss: 0.281) - Steps:  20%|█▉        | 743/3750 [1:11:48<4:50:36,  5.80s/it]


KeyboardInterrupt: 

In [ ]:
# MLflow 3: `register_model` expects `mlflow.*.log_model` (LoggedModel + MLmodel). Raw
# `log_artifacts` from save_pretrained does not qualify; use the registry client instead.
# model_uri = f"runs:/{run.info.run_id}/model"
# _mlfc.create_model_version(
#     name="agnews-transformer",
#     source=model_uri,
#     run_id=run.info.run_id,
# )
